In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CODEBASE_DIR = Path('/content/drive/MyDrive/predicting-the-future-effect-of-optimizer-memory')  
assert (CODEBASE_DIR / 'requirements.txt').is_file(), 'Upload and unzip the code folder to Drive first.'
os.chdir(CODEBASE_DIR)
sys.path.insert(0, str(CODEBASE_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


In [ ]:
from rttp_grid.candidate_selection import CandidateSelectionConfig

ROOT_NAME = 'results/rttp_cifar_candidate_selection'
RUN_NAME = 'rttp_cifar_candidate_selection'

ARCHITECTURE = 'smallcnn'  
SEEDS = (0,)               
HORIZON = 40
LEARNING_RATE = 0.03
MOMENTUM = 0.9
RADIUS_SCALE = 1e-3

RESPONSE_RANK = 16
CANDIDATE_COUNT = 100
OBJECTIVE_TRADEOFFS = (0.0, 0.25, 0.5, 1.0)
TOPK_VALUES = (5, 10)


EVAL_BATCHES_PER_TASK = 0
FORCE_BOUNDARY = False

cfg = CandidateSelectionConfig(
    run_name=RUN_NAME,
    root_name=ROOT_NAME,
    architecture=ARCHITECTURE,
    seeds=SEEDS,
    horizon=HORIZON,
    learning_rate=LEARNING_RATE,
    momentum=MOMENTUM,
    radius_scale=RADIUS_SCALE,
    response_rank=RESPONSE_RANK,
    candidate_count=CANDIDATE_COUNT,
    objective_tradeoffs=OBJECTIVE_TRADEOFFS,
    topk_values=TOPK_VALUES,
    eval_batches_per_task=EVAL_BATCHES_PER_TASK,
    force_boundary=FORCE_BOUNDARY,
)
cfg

In [ ]:
from rttp_grid.candidate_selection import run_candidate_selection_experiment

output_root = run_candidate_selection_experiment(cfg)
print('output_root:', output_root)

In [ ]:
import pandas as pd
from pathlib import Path

from rttp_grid.utils import get_root
root = get_root(cfg)
metrics = pd.read_csv(root / 'tables' / 'candidate_ranking_metrics.csv')
summary = pd.read_csv(root / 'tables' / 'candidate_ranking_summary.csv')
timing = pd.read_csv(root / 'tables' / 'candidate_selection_timing.csv')
quality = pd.read_csv(root / 'tables' / 'candidate_endpoint_prediction_quality.csv')

display(summary)
display(metrics)
display(timing)
display(quality.groupby('predictor').mean(numeric_only=True).reset_index())

In [ ]:
import matplotlib.pyplot as plt

scores = pd.read_csv(root / 'tables' / 'candidate_scores_by_objective.csv')
fig_dir = root / 'figures'
fig_dir.mkdir(exist_ok=True)

for tradeoff in cfg.objective_tradeoffs:
    true = scores[(scores.predictor == 'true_replay') & (scores.tradeoff == tradeoff)]
    for predictor in ['time_varying_rttp', 'frozen_boundary']:
        pred = scores[(scores.predictor == predictor) & (scores.tradeoff == tradeoff)]
        merged = true.merge(pred, on=['seed', 'candidate_id', 'tradeoff'], suffixes=('_true', '_pred'))
        plt.figure(figsize=(5.4, 4.8))
        plt.scatter(merged['score_pred'], merged['score_true'], s=22, alpha=0.75)
        lo = min(merged['score_pred'].min(), merged['score_true'].min())
        hi = max(merged['score_pred'].max(), merged['score_true'].max())
        plt.plot([lo, hi], [lo, hi], '--', color='black', linewidth=1)
        plt.xlabel('predicted score')
        plt.ylabel('true replay score')
        plt.title(f'{predictor}, lambda={tradeoff:g}')
        plt.tight_layout()
        stem = f'candidate_score_scatter_{predictor}_lambda={tradeoff:g}'.replace('.', 'p')
        plt.savefig(fig_dir / f'{stem}.png', dpi=220)
        plt.savefig(fig_dir / f'{stem}.pdf')
        plt.show()

In [ ]:
table = summary.copy()
display(table)
print((root / 'tables' / 'candidate_ranking_summary.tex').read_text())

In [ ]:
import shutil

archive = shutil.make_archive(str(root), 'zip', root)
print('archive:', archive)